In [1]:
import numpy as np
from sklearn.datasets import make_blobs

def make_data(seed=0):
    X, y = make_blobs(
        n_samples=150,
        centers=3,
        n_features=2,
        cluster_std=1.0,
        random_state=seed
    )
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    return X, y

def forward(X, W, b):
    return X @ W + b

def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def loss_and_grads(X, y, W, b):
    N = X.shape[0]
    logits = forward(X, W, b)
    probs = softmax(logits)

    correct_probs = np.clip(probs[np.arange(N), y], 1e-12, 1.0)
    loss = float(-np.mean(np.log(correct_probs)))

    dZ = probs.copy()
    dZ[np.arange(N), y] -= 1.0
    dZ /= N

    dW = X.T @ dZ
    db = dZ.sum(axis=0)
    return loss, dW, db

def sgd_step(W, b, dW, db, lr):
    return W - lr * dW, b - lr * db

def train(X, y, lr, epochs, seed=0):
    D = X.shape[1]
    K = int(np.max(y)) + 1
    W = np.zeros((D, K))
    b = np.zeros(K)
    losses = []

    for _ in range(epochs):
        loss, dW, db = loss_and_grads(X, y, W, b)
        losses.append(loss)
        W, b = sgd_step(W, b, dW, db, lr)

    return W, b, losses

def lr_effect(X, y, lrs, epochs=200, seed=0):
    result = {}
    for lr in lrs:
        W, b, losses = train(X, y, lr, epochs, seed)
        if np.isfinite(losses[-1]) and losses[-1] < losses[0]:
            result[lr] = "converged"
        else:
            result[lr] = "diverged"
    return result


In [2]:
X, y = make_data()
W, b, losses = train(X, y, 0.5, 300)
print(round(losses[0], 3), "->", round(losses[-1], 3))
print(lr_effect(X, y, [0.01, 0.3, 1000.0]))


1.099 -> 0.165
{0.01: 'converged', 0.3: 'converged', 1000.0: 'diverged'}
